In [ ]:
# SMARANA synthetic demo training
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy==2.1.3", "pandas==2.2.3", "scikit-learn==1.6.1", "joblib==1.4.2"], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'numpy==2.1.3', 'pandas==2.2.3', 'scikit-learn==1.6.1', 'joblib==1.4.2'], returncode=0)

In [ ]:
# Synthetic demo only: 120 simulated users, 12 games, 16 sessions/game.
# Corrected smarana-history-v1 features. No real patient data.
import random, json, statistics, hashlib, platform, zipfile
from pathlib import Path
from datetime import datetime, timedelta, timezone
from collections import Counter
import numpy as np, pandas as pd, sklearn, joblib
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, f1_score
ROOT=Path('/content/smarana-dda-demo'); ROOT.mkdir(exist_ok=True)
OUT=ROOT/'artifacts'; OUT.mkdir(exist_ok=True)
GAMES=['sequence_recall','memory_match','find_the_change','object_sorting','daily_routine','word_recall','visual_search','pattern_completion','spatial_recall','attention_tap','association_game','personal_memory']
rng=random.Random(42); rows=[]; origin=datetime(2026,1,1,tzinfo=timezone.utc)
def clip(x,lo,hi): return max(lo,min(hi,x))
for patient in range(120):
    ability=rng.uniform(-.25,.25); condition=rng.choice(['none','none','none','low_vision','motor_tremor'])
    for gi,game in enumerate(GAMES):
        level=rng.randint(1,3); history=[]
        for si in range(16):
            fatigue=max(0,si-10)*.012
            accuracy=clip(.77+ability-(level-1)*.11-fatigue+rng.gauss(0,.11),.05,1.)
            early=accuracy<.42 or rng.random()<(.015+fatigue/3)
            rounds=rng.randint(3,6); hints=max(0,int(round((1-accuracy)*rounds+rng.gauss(0,.6))))
            rt=int(clip(1500+level*380-ability*800+fatigue*1500+rng.gauss(0,260),500,6000))
            event={'difficulty':level,'accuracy':round(accuracy,4),'reaction_time_ms':rt,'hints_used':hints,'rounds_completed':0 if early else rounds,'early_exit':early,'session_duration_sec':int(clip(rounds*rt/1000*1.8,20,600)),'errors':max(0,rounds-int(round(accuracy*rounds)))}
            target=-1 if early or accuracy<.55 else (1 if len(history[-3:])==3 and all(a>=.85 for a in history[-3:]) and accuracy>=.85 else 0)
            next_level=int(clip(level+target,1,5))
            rows.append({'patient_id':f'synthetic-{patient:03d}','game_id':game,'timestamp':(origin+timedelta(days=si,minutes=gi)).isoformat(),'performance':event,'condition':condition,'applied_adjustment':next_level-level,'target_adj':target,'provenance':'synthetic-demo-policy-v1'})
            history.append(accuracy); level=next_level
FEATURE_COLUMNS=['current_difficulty','accuracy','response_time','hints_used','early_exit','rounds_this_session','session_duration','rolling_accuracy_3','rolling_response_time_3','accuracy_trend','response_time_trend','recent_hint_rate','consecutive_errors','consecutive_successes','previous_adjustment','difficulty_change_count_5','recent_early_exit_rate','is_cold_start','condition']
train,test=next(GroupShuffleSplit(test_size=.2,random_state=42).split(rows,groups=[r['patient_id'] for r in rows]))
neutral=statistics.median(rows[int(i)]['performance']['reaction_time_ms']/1000 for i in train)
def build_features(current,history,rt_neutral,condition):
    past=history[-6:]; recent=past[-3:]; enough=len(recent)==3
    def mean(items,key): return float(sum(r[key] for r in items)/len(items))
    acc=mean(recent,'accuracy') if enough else .65
    rt=mean(recent,'response_time') if enough else rt_neutral
    changes=[r.get('adjustment',0) for r in history[-5:]]
    def streak(predicate):
        count=0
        for r in reversed(history):
            if not predicate(r['accuracy']): break
            count+=1
        return count
    values=[.1+(current['difficulty']-1)*.225,current['accuracy'],current['reaction_time_ms']/1000,current['hints_used'],int(current['early_exit']),current['rounds_completed'],current['session_duration_sec'],acc,rt,acc-mean(past[:3],'accuracy') if len(past)==6 else 0,rt-mean(past[:3],'response_time') if len(past)==6 else 0,sum(r['hints_used'] for r in recent)/max(1,sum(r['rounds'] for r in recent)) if enough else 0,streak(lambda a:a<.55),streak(lambda a:a>.75),changes[-1] if changes else 0,sum(v!=0 for v in changes) if len(changes)==5 else 0,sum(r['early_exit'] for r in history[-5:])/5 if len(history)>=5 else 0,int(len(history)<3),condition]
    return dict(zip(FEATURE_COLUMNS,values))
histories={}; features=[None]*len(rows); prior=[None]*len(rows)
for i in sorted(range(len(rows)),key=lambda i:rows[i]['timestamp']):
    r=rows[i]; p=r['performance']; h=histories.setdefault((r['patient_id'],r['game_id']),[])
    prior[i]=list(h); features[i]=build_features(p,h,neutral,r['condition'])
    h.append({'accuracy':p['accuracy'],'response_time':p['reaction_time_ms']/1000,'hints_used':p['hints_used'],'rounds':p['rounds_completed'],'early_exit':p['early_exit'],'adjustment':r['applied_adjustment']})
frame=pd.DataFrame(features,columns=FEATURE_COLUMNS); labels=np.array([r['target_adj'] for r in rows])
assert set(labels[train])=={-1,0,1}
train_users={rows[int(i)]['patient_id'] for i in train}; test_users={rows[int(i)]['patient_id'] for i in test}
assert train_users.isdisjoint(test_users)
pipeline=Pipeline([('prep',ColumnTransformer([('cat',OneHotEncoder(handle_unknown='ignore',sparse_output=False),['condition']),('num','passthrough',FEATURE_COLUMNS[:-1])],verbose_feature_names_out=False)),('clf',RandomForestClassifier(n_estimators=200,random_state=42,class_weight='balanced_subsample',n_jobs=-1))])
print('Training:',len(rows),'synthetic rows;',len(train_users),'train users;',len(test_users),'held-out users; classes',dict(Counter(labels.tolist())),flush=True)
pipeline.fit(frame.iloc[train],labels[train]); pred=pipeline.predict(frame.iloc[test])
schema={key:{'min':min(0,float(frame.iloc[train][key].min())),'max':max(1,float(frame.iloc[train][key].max())*1.1)} for key in FEATURE_COLUMNS[:-1]}
schema['accuracy']={'min':0,'max':1}; schema['current_difficulty']={'min':.1,'max':1}
score=float(f1_score(labels[test],pred,average='macro'))
metadata={'model_version':'synthetic-demo-cloud-v1','feature_contract':'smarana-history-v1','feature_columns':FEATURE_COLUMNS,'feature_schema':schema,'rt_neutral':neutral,'library_versions':{'sklearn':sklearn.__version__,'pandas':pd.__version__,'numpy':np.__version__},'metrics':{'held_out_user_macro_f1':score},'limitations':'Synthetic policy imitation only. Not clinically validated.'}
artifact_path=OUT/'dda-synthetic-demo-v1.joblib'
joblib.dump({'pipeline':pipeline,'metadata':metadata},artifact_path,compress=3)
loaded=joblib.load(artifact_path)['pipeline']; cases=frame.iloc[test[:5]]
assert np.array_equal(pipeline.predict(cases),loaded.predict(cases))
parity=[{'features':f,'prediction':int(p)} for f,p in zip(cases.to_dict('records'),loaded.predict(cases))]
artifact_path.with_suffix('.parity.json').write_text(json.dumps(parity,indent=2))
# Save full runtime feature-parity cases: current + prior history, not just vectors.
feature_cases=[{'current':rows[int(i)]['performance'],'history':prior[int(i)],'condition':rows[int(i)]['condition'],'rt_neutral':neutral,'features':features[int(i)]} for i in test[:20]]
(OUT/'feature-parity.json').write_text(json.dumps(feature_cases,indent=2))
report={'repository':'https://gitlab.com/smarana-group1/smarana','source_commit':'5d6dff35bbff0a4792e15137b620b07e04bdb62d','runtime':'Google Colab Google Compute Engine CPU','python':platform.python_version(),'provenance':'synthetic-demo-policy-v1','seed':42,'rows':len(rows),'train_users':len(train_users),'test_users':len(test_users),'user_overlap':len(train_users & test_users),'held_out_user_macro_f1':score,'classification_report':classification_report(labels[test],pred,labels=[-1,0,1],output_dict=True,zero_division=0),'confusion_matrix_labels':[-1,0,1],'confusion_matrix':confusion_matrix(labels[test],pred,labels=[-1,0,1]).tolist(),'reload_parity_passed':True,'artifact_sha256':hashlib.sha256(artifact_path.read_bytes()).hexdigest(),'library_versions':metadata['library_versions'],'limitations':'Synthetic policy imitation only; no real patients or clinical validation. Applied actions reflect bounded actual level changes.'}
(OUT/'training-report.json').write_text(json.dumps(report,indent=2))
(OUT/'synthetic-dda-events.json').write_text(json.dumps(rows))
print(json.dumps(report,indent=2))
print('MODEL_TRAINING_AND_EXPORT_PASSED')


Training: 23040 synthetic rows; 96 train users; 24 held-out users; classes {0: 18257, -1: 4484, 1: 299}
{
  "repository": "https://gitlab.com/smarana-group1/smarana",
  "source_commit": "5d6dff35bbff0a4792e15137b620b07e04bdb62d",
  "runtime": "Google Colab Google Compute Engine CPU",
  "python": "3.13.15",
  "provenance": "synthetic-demo-policy-v1",
  "seed": 42,
  "rows": 23040,
  "train_users": 96,
  "test_users": 24,
  "user_overlap": 0,
  "held_out_user_macro_f1": 0.9450115838365466,
  "classification_report": {
    "-1": {
      "precision": 1.0,
      "recall": 1.0,
      "f1-score": 1.0,
      "support": 856.0
    },
    "0": {
      "precision": 0.9970230040595399,
      "recall": 0.9978331527627302,
      "f1-score": 0.9974279139028022,
      "support": 3692.0
    },
    "1": {
      "precision": 0.8596491228070176,
      "recall": 0.8166666666666667,
      "f1-score": 0.8376068376068376,
      "support": 60.0
    },
    "accuracy": 0.9958767361111112,
    "macro avg": {
     

In [ ]:
# Export artifact, evaluation and feature/prediction parity cases.
from google.colab import files
bundle=ROOT/'smarana-dda-synthetic-demo-cloud-v1.zip'
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUT.glob('*')):
        if path.name != 'synthetic-dda-events.json': z.write(path, 'artifacts/'+path.name)
print('Export bundle:',bundle.name,'bytes:',bundle.stat().st_size)
files.download(str(bundle))

Export bundle: smarana-dda-synthetic-demo-cloud-v1.zip bytes: 2812328


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Exact backend feature-builder semantics, checked across every synthetic session.
import math
def backend_build_features(current, history, rt_neutral, condition):
    if condition not in {'none','low_vision','motor_tremor'}: raise ValueError('Accessibility context has not been collected')
    if not math.isfinite(rt_neutral) or rt_neutral <= 0: raise ValueError('Missing training RT baseline')
    past=history[-6:]; recent=past[-3:]; enough=len(recent)==3
    def mean(rows,key): return float(sum(row[key] for row in rows)/len(rows))
    acc=mean(recent,'accuracy') if enough else .65
    rt=mean(recent,'response_time') if enough else rt_neutral
    changes=[row.get('adjustment',0) for row in history[-5:]]
    def streak(predicate):
        count=0
        for row in reversed(history):
            if not predicate(row['accuracy']): break
            count+=1
        return count
    result={'current_difficulty':.1+(current['difficulty']-1)*.225,'accuracy':current['accuracy'],'response_time':current['reaction_time_ms']/1000,'hints_used':current['hints_used'],'early_exit':int(current['early_exit']),'rounds_this_session':current['rounds_completed'],'session_duration':current['session_duration_sec'],'rolling_accuracy_3':acc,'rolling_response_time_3':rt,'accuracy_trend':acc-mean(past[:3],'accuracy') if len(past)==6 else 0,'response_time_trend':rt-mean(past[:3],'response_time') if len(past)==6 else 0,'recent_hint_rate':sum(row['hints_used'] for row in recent)/max(1,sum(row['rounds'] for row in recent)) if enough else 0,'consecutive_errors':streak(lambda accuracy:accuracy<.55),'consecutive_successes':streak(lambda accuracy:accuracy>.75),'previous_adjustment':changes[-1] if changes else 0,'difficulty_change_count_5':sum(value!=0 for value in changes) if len(changes)==5 else 0,'recent_early_exit_rate':sum(row['early_exit'] for row in history[-5:])/5 if len(history)>=5 else 0,'is_cold_start':int(len(history)<3),'condition':condition}
    if any(not math.isfinite(value) for key,value in result.items() if key!='condition'): raise ValueError('Non-finite feature')
    return {key:result[key] for key in FEATURE_COLUMNS}
for i,row in enumerate(rows):
    assert backend_build_features(row['performance'],prior[i],neutral,row['condition']) == features[i], i
# Backend runtime inference, reproduced from notebook_dda.recommend.
def backend_recommend(current,history,condition):
    if len(history)<3: return {'adjustment':0,'reason':'cold_start'}
    model=joblib.load(artifact_path); meta=model['metadata']
    assert meta['feature_columns']==FEATURE_COLUMNS and meta['feature_contract']=='smarana-history-v1'
    assert meta['library_versions']['sklearn']==sklearn.__version__
    f=backend_build_features(current,history,meta['rt_neutral'],condition)
    for key in FEATURE_COLUMNS[:-1]:
        spec=meta['feature_schema'][key]
        if not spec['min']<=f[key]<=spec['max']: return {'adjustment':0,'reason':'model_unavailable'}
    x=pd.DataFrame([f],columns=FEATURE_COLUMNS)
    confidence=float(max(model['pipeline'].predict_proba(x)[0])); prediction=int(model['pipeline'].predict(x)[0])
    assert prediction in {-1,0,1}
    adjustment=prediction if confidence>=.6 else 0
    if current['early_exit'] or current['accuracy']<.55: adjustment=-1
    if adjustment>0 and (f['consecutive_errors']>0 or f['recent_early_exit_rate']>.2 or current['errors']>=max(1,current['rounds_completed']/2)): adjustment=0
    flags=[r.get('adjustment',0) for r in history[-10:]]
    last_change=next((index for index,value in enumerate(reversed(flags)) if value),None)
    if sum(v!=0 for v in flags)>=4: adjustment=0
    elif adjustment==1 and last_change is not None and last_change<3: adjustment=0
    elif adjustment==1 and history[-1].get('raw_prediction')!=1: adjustment=0
    elif adjustment==-1 and last_change==0 and not current['early_exit']: adjustment=0
    return {'adjustment':adjustment,'raw_prediction':prediction,'engine_version':'notebook-rf-v1','model_version':meta['model_version'],'reason':'model'}
examples=[]
for i in test:
    i=int(i)
    if len(prior[i])>=6:
        decision=backend_recommend(rows[i]['performance'],prior[i],rows[i]['condition'])
        if decision['reason']=='model': examples.append(decision)
    if len(examples)==5: break
assert len(examples)==5 and all(e['adjustment'] in [-1,0,1] for e in examples)
report['backend_feature_parity_rows']=len(rows)
report['backend_runtime_examples']=examples
report['backend_cold_start_example']=backend_recommend(rows[0]['performance'],[],rows[0]['condition'])
(OUT/'training-report.json').write_text(json.dumps(report,indent=2))
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUT.glob('*')):
        if path.name!='synthetic-dda-events.json': z.write(path,'artifacts/'+path.name)
print('BACKEND_FEATURE_PARITY_PASSED:',len(rows),'rows')
print('BACKEND_RUNTIME_INFERENCE_PASSED:',json.dumps(examples))
print('EXPORT_BUNDLE_READY:',str(bundle))
# A compact case enables independent local builder verification.
print('LOCAL_FEATURE_CASE:',json.dumps(feature_cases[6]))


BACKEND_FEATURE_PARITY_PASSED: 23040 rows
BACKEND_RUNTIME_INFERENCE_PASSED: [{"adjustment": -1, "raw_prediction": -1, "engine_version": "notebook-rf-v1", "model_version": "synthetic-demo-cloud-v1", "reason": "model"}, {"adjustment": 0, "raw_prediction": 0, "engine_version": "notebook-rf-v1", "model_version": "synthetic-demo-cloud-v1", "reason": "model"}, {"adjustment": 0, "raw_prediction": 0, "engine_version": "notebook-rf-v1", "model_version": "synthetic-demo-cloud-v1", "reason": "model"}, {"adjustment": 0, "raw_prediction": 0, "engine_version": "notebook-rf-v1", "model_version": "synthetic-demo-cloud-v1", "reason": "model"}, {"adjustment": 0, "raw_prediction": 0, "engine_version": "notebook-rf-v1", "model_version": "synthetic-demo-cloud-v1", "reason": "model"}]
EXPORT_BUNDLE_READY: /content/smarana-dda-demo/smarana-dda-synthetic-demo-cloud-v1.zip
LOCAL_FEATURE_CASE: {"current": {"difficulty": 3, "accuracy": 0.3315, "reaction_time_ms": 2774, "hints_used": 4, "rounds_completed": 0, "ea

In [ ]:
# Store a self-contained model download in the saved notebook output.
import base64
from IPython.display import display, HTML
encoded_bundle=base64.b64encode(bundle.read_bytes()).decode('ascii')
display(HTML('<p>Synthetic demo model, evaluation report and parity checks:</p><a download="smarana-dda-synthetic-demo-cloud-v1.zip" href="data:application/zip;base64,'+encoded_bundle+'">Download trained SMARANA demo model bundle</a>'))
print('Embedded export bytes:',bundle.stat().st_size)


Embedded export bytes: 2812470
